# Individual Task 1 – Part 1.3: Data Analysis

## Machine Learning Analysis of Epilepsy EEG and Stroke Prediction Data

**Student:** Himasri Madala  
**Student ID:** S4222141  
**Course:** Master of Data Science  
**University:** RMIT University

### Objective

This analysis applies two machine learning algorithms, Support Vector Machine (SVM) and Multi-Layer Perceptron (MLP) neural network, to two publicly available healthcare datasets:

1. BEED (Bangalore EEG Epilepsy Dataset)
2. Stroke Prediction Dataset

The aim is to investigate whether machine learning models can identify patterns associated with neurological and clinical outcomes and to compare their effectiveness using appropriate evaluation metrics.

The analysis follows a leakage-free machine learning workflow. The data are first divided into training and testing sets, after which preprocessing transformations are fitted only on the training data and applied to both subsets.

## 1. Import Required Libraries

The following Python libraries are used for:

- Loading and manipulating datasets
- Numerical computation
- Splitting data into training and testing sets
- Encoding categorical variables
- Imputing missing values
- Feature scaling
- Training SVM and MLP models
- Evaluating classification performance

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder,
    OneHotEncoder
)

from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.impute import SimpleImputer

## 2. Dataset File Paths and Reproducibility

The following file paths specify the locations of the two datasets.

A fixed random state of 42 is used throughout the analysis to make the train-test split and model results reproducible.

In [2]:
# File paths
BEED_PATH = "beed_+bangalore+eeg+epilepsy+dataset/BEED_Data.csv"
STROKE_PATH = "healthcare-dataset-stroke-data.csv"

# Random state for reproducibility
RANDOM_STATE = 42

## 3. Model Evaluation Function

The models are evaluated using several classification metrics:

- **Accuracy:** proportion of observations classified correctly.
- **Precision:** proportion of predicted positive/classified cases that are correct.
- **Recall:** proportion of actual cases correctly identified.
- **F1-score:** harmonic mean of precision and recall.
- **ROC-AUC:** measures the model's ability to discriminate between classes.
- **Confusion matrix:** shows correct and incorrect predictions for each class.

Weighted averages are used for precision, recall and F1-score so that differences in class sizes are taken into account.

In [3]:
def evaluate_model(name, y_test, y_pred, y_proba=None):
    """Print a consistent block of evaluation metrics."""
    
    print(f"\n--- {name} ---")
    
    print(
        "Accuracy :",
        round(accuracy_score(y_test, y_pred), 4)
    )
    
    print(
        "Precision:",
        round(
            precision_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            ),
            4
        )
    )
    
    print(
        "Recall   :",
        round(
            recall_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            ),
            4
        )
    )
    
    print(
        "F1 Score :",
        round(
            f1_score(
                y_test,
                y_pred,
                average="weighted",
                zero_division=0
            ),
            4
        )
    )
    
    # ROC-AUC
    if y_proba is not None:
        try:
            if y_proba.ndim == 2 and y_proba.shape[1] == 2:
                auc = roc_auc_score(
                    y_test,
                    y_proba[:, 1]
                )
            else:
                auc = roc_auc_score(
                    y_test,
                    y_proba,
                    multi_class="ovr"
                )
            
            print("ROC-AUC  :", round(auc, 4))
            
        except Exception as e:
            print(
                "ROC-AUC  : could not compute (",
                e,
                ")"
            )
    
    print(
        "Confusion Matrix:\n",
        confusion_matrix(y_test, y_pred)
    )
    
    print(
        "\nFull classification report:\n",
        classification_report(
            y_test,
            y_pred,
            zero_division=0
        )
    )

# 4. Dataset 1 – BEED Epilepsy EEG Dataset

The first dataset used in this analysis is the BEED (Bangalore EEG Epilepsy Dataset).

The dataset contains EEG-related numerical features that can be used to classify epilepsy-related patterns.

This dataset is directly relevant to the selected Junior Data Scientist role at the Australian Epilepsy Project because the role involves applying machine learning to healthcare data and contributing to epilepsy research.

The dataset is analysed as a classification problem.

## 4.1 Loading the BEED Dataset

The dataset is loaded from a CSV file and its dimensions and column names are inspected.

In [4]:
print("=" * 60)
print("LOADING BEED DATASET")
print("=" * 60)

beed = pd.read_csv(BEED_PATH)

print("Shape:", beed.shape)

print("\nColumns:")
print(list(beed.columns))

print("\nFirst five rows:")
display(beed.head())

LOADING BEED DATASET
Shape: (8000, 17)

Columns:
['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8', 'X9', 'X10', 'X11', 'X12', 'X13', 'X14', 'X15', 'X16', 'y']

First five rows:


,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,X11,X12,X13,X14,X15,X16,y
0,4,7,18,25,28,27,20,10,-10,-18,-20,-16,13,32,12,10,0
1,87,114,120,106,76,54,28,5,-19,-49,-85,-102,-100,-89,-61,-21,0
2,-131,-133,-140,-131,-123,-108,-58,-51,-70,-77,-76,-76,-73,-57,-40,-14,0
3,68,104,73,34,-12,-26,-38,-36,-67,-88,-25,31,18,-4,6,-29,0
4,-67,-90,-97,-94,-86,-71,-43,-11,23,46,58,50,39,19,-9,-41,0


## 4.2 Initial Dataset Inspection

The target variable and feature structure are examined before preprocessing.

In [5]:
# Identify target column
label_col = "y" if "y" in beed.columns else beed.columns[-1]

# Separate features and target
X_beed = beed.drop(columns=[label_col])

y_beed = beed[label_col]

# Keep numeric features only
X_beed = X_beed.select_dtypes(
    include=[np.number]
)

print("Target column:", label_col)

print("\nFeature shape:", X_beed.shape)

print("\nTarget distribution:")
print(y_beed.value_counts())

print("\nUnique target values:")
print(sorted(y_beed.unique()))

Target column: y

Feature shape: (8000, 16)

Target distribution:
y
0    2000
1    2000
2    2000
3    2000
Name: count, dtype: int64

Unique target values:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


## 4.3 Train-Test Split

The BEED dataset is divided into:

- 80% training data
- 20% testing data

Stratified sampling is used to preserve the distribution of target classes in both subsets.

The split is performed before imputation and scaling to prevent information from the test set influencing the preprocessing process.

In [6]:
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_beed,
    y_beed,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_beed
)

print("Training observations:", X_train_b.shape[0])
print("Testing observations :", X_test_b.shape[0])

Training observations: 6400
Testing observations : 1600


## 4.4 Handling Missing Values

Missing numerical values are replaced using mean imputation.

The imputer is fitted only on the training data and then applied to both the training and testing data. This prevents data leakage from the test set.

In [7]:
imputer_beed = SimpleImputer(
    strategy="mean"
)

X_train_b = pd.DataFrame(
    imputer_beed.fit_transform(X_train_b),
    columns=X_beed.columns
)

X_test_b = pd.DataFrame(
    imputer_beed.transform(X_test_b),
    columns=X_beed.columns
)

print(
    "Missing values in training data:",
    X_train_b.isnull().sum().sum()
)

print(
    "Missing values in testing data:",
    X_test_b.isnull().sum().sum()
)

Missing values in training data: 0
Missing values in testing data: 0


## 4.5 Feature Scaling

Standardisation is applied because both SVM and MLP are sensitive to differences in feature scales.

The scaler is fitted only on the training data and then used to transform both training and testing data.

In [8]:
scaler_beed = StandardScaler()

X_train_b_scaled = scaler_beed.fit_transform(
    X_train_b
)

X_test_b_scaled = scaler_beed.transform(
    X_test_b
)

print(
    "Scaled training shape:",
    X_train_b_scaled.shape
)

print(
    "Scaled testing shape:",
    X_test_b_scaled.shape
)

Scaled training shape: (6400, 16)
Scaled testing shape: (1600, 16)


## 4.6 Support Vector Machine (SVM) – BEED

A Support Vector Machine with a radial basis function (RBF) kernel is trained on the BEED dataset.

The RBF kernel allows the model to learn nonlinear decision boundaries between the different EEG-related classes.

Probability estimates are enabled so that ROC-AUC can also be calculated.

In [9]:
svm_beed = SVC(
    kernel="rbf",
    probability=True,
    random_state=RANDOM_STATE
)

svm_beed.fit(
    X_train_b_scaled,
    y_train_b
)

pred_svm_b = svm_beed.predict(
    X_test_b_scaled
)

proba_svm_b = svm_beed.predict_proba(
    X_test_b_scaled
)

evaluate_model(
    "SVM on BEED",
    y_test_b,
    pred_svm_b,
    proba_svm_b
)


--- SVM on BEED ---
Accuracy : 0.7444
Precision: 0.7731
Recall   : 0.7444
F1 Score : 0.731
ROC-AUC  : 0.9272
Confusion Matrix:
 [[393   5   0   2]
 [  0 284  77  39]
 [  0  13 373  14]
 [  0  50 209 141]]

Full classification report:
               precision    recall  f1-score   support

           0       1.00      0.98      0.99       400
           1       0.81      0.71      0.76       400
           2       0.57      0.93      0.70       400
           3       0.72      0.35      0.47       400

    accuracy                           0.74      1600
   macro avg       0.77      0.74      0.73      1600
weighted avg       0.77      0.74      0.73      1600



## 4.7 Multi-Layer Perceptron (MLP) – BEED

A Multi-Layer Perceptron neural network is used as the second machine learning algorithm.

The network contains two hidden layers with 64 and 32 neurons respectively.

Early stopping is enabled to reduce unnecessary training and help limit overfitting.

In [10]:
mlp_beed = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    early_stopping=True,
    random_state=RANDOM_STATE
)

mlp_beed.fit(
    X_train_b_scaled,
    y_train_b
)

pred_mlp_b = mlp_beed.predict(
    X_test_b_scaled
)

proba_mlp_b = mlp_beed.predict_proba(
    X_test_b_scaled
)

evaluate_model(
    "MLP (Neural Network) on BEED",
    y_test_b,
    pred_mlp_b,
    proba_mlp_b
)


--- MLP (Neural Network) on BEED ---
Accuracy : 0.9419
Precision: 0.9428
Recall   : 0.9419
F1 Score : 0.9422
ROC-AUC  : 0.9939
Confusion Matrix:
 [[398   0   0   2]
 [  0 381   6  13]
 [  0   1 367  32]
 [  0   5  34 361]]

Full classification report:
               precision    recall  f1-score   support

           0       1.00      0.99      1.00       400
           1       0.98      0.95      0.97       400
           2       0.90      0.92      0.91       400
           3       0.88      0.90      0.89       400

    accuracy                           0.94      1600
   macro avg       0.94      0.94      0.94      1600
weighted avg       0.94      0.94      0.94      1600



# 5. Dataset 2 – Stroke Prediction Dataset

The second dataset contains demographic and clinical characteristics associated with stroke occurrence.

Unlike the BEED dataset, which contains EEG-related features, the Stroke Prediction Dataset contains structured patient-level clinical and demographic information.

This provides a complementary data source for evaluating machine learning models on healthcare data.

## 5.1 Loading the Stroke Dataset

The Stroke Prediction Dataset is loaded from a CSV file and its dimensions and columns are inspected.

In [11]:
print("=" * 60)
print("LOADING STROKE DATASET")
print("=" * 60)

stroke = pd.read_csv(STROKE_PATH)

print("Shape:", stroke.shape)

print("\nColumns:")
print(list(stroke.columns))

print("\nFirst five rows:")
display(stroke.head())

LOADING STROKE DATASET
Shape: (5110, 12)

Columns:
['id', 'gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']

First five rows:


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 5.2 Initial Dataset Inspection

The dataset is inspected to identify the target variable, categorical variables and missing values.

In [12]:
print("Data types:")
print(stroke.dtypes)

print("\nMissing values:")
print(stroke.isnull().sum())

Data types:
id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

Missing values:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


## 5.3 Feature and Target Selection

The `id` column is removed because it is an identifier and does not contain meaningful predictive information.

The target variable is `stroke`, indicating whether a patient experienced a stroke.

In [13]:
# Remove identifier column
if "id" in stroke.columns:
    stroke = stroke.drop(columns=["id"])

# Separate target and predictors
y_stroke = stroke["stroke"]

X_stroke = stroke.drop(
    columns=["stroke"]
)

print("Feature shape:", X_stroke.shape)

print("\nTarget distribution:")
print(y_stroke.value_counts())

print("\nTarget proportions:")
print(
    y_stroke.value_counts(
        normalize=True
    )
)

Feature shape: (5110, 10)

Target distribution:
stroke
0    4861
1     249
Name: count, dtype: int64

Target proportions:
stroke
0    0.951272
1    0.048728
Name: proportion, dtype: float64


## 5.4 Identifying Variable Types

The predictor variables are separated into:

- Binary categorical variables
- Nominal categorical variables
- Numerical variables

Binary variables are represented using 0/1 encoding, while nominal variables with more than two categories are one-hot encoded.

In [14]:
binary_cols = [
    c for c in [
        "gender",
        "ever_married",
        "Residence_type"
    ]
    if c in X_stroke.columns
]

nominal_cols = [
    c for c in [
        "work_type",
        "smoking_status"
    ]
    if c in X_stroke.columns
]

numeric_cols = [
    c for c in X_stroke.columns
    if c not in binary_cols + nominal_cols
]

print("Binary columns:")
print(binary_cols)

print("\nNominal columns:")
print(nominal_cols)

print("\nNumeric columns:")
print(numeric_cols)

Binary columns:
['gender', 'ever_married', 'Residence_type']

Nominal columns:
['work_type', 'smoking_status']

Numeric columns:
['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']


## 5.5 Train-Test Split

The Stroke Prediction Dataset is divided into 80% training data and 20% testing data.

Stratification is used because the target variable is highly imbalanced, with substantially fewer stroke cases than non-stroke cases.

The split is performed before preprocessing to prevent data leakage.

In [15]:
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_stroke,
    y_stroke,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_stroke
)

print("Training observations:", X_train_s.shape[0])
print("Testing observations :", X_test_s.shape[0])

Training observations: 4088
Testing observations : 1022


## 5.6 Encoding Binary Variables

Binary categorical variables are converted to 0/1 representations.

The encoding is fitted using the training data only. Any category that unexpectedly occurs in the test set but was not present during training is mapped to the first known category.

In [16]:
X_train_s = X_train_s.copy()
X_test_s = X_test_s.copy()

for col in binary_cols:
    
    le = LabelEncoder()
    
    le.fit(
        X_train_s[col].astype(str)
    )
    
    X_train_s[col] = le.transform(
        X_train_s[col].astype(str)
    )
    
    known = set(le.classes_)
    
    X_test_s[col] = (
        X_test_s[col]
        .astype(str)
        .apply(
            lambda v:
            v if v in known
            else le.classes_[0]
        )
    )
    
    X_test_s[col] = le.transform(
        X_test_s[col]
    )

print("Binary encoding completed.")

Binary encoding completed.


## 5.7 One-Hot Encoding of Nominal Variables

Nominal variables such as `work_type` and `smoking_status` do not have a meaningful numerical order.

One-hot encoding is therefore used to represent each category as a separate binary feature.

The encoder is fitted only on the training data and `handle_unknown="ignore"` ensures that unseen categories in the test set do not cause errors.

In [17]:
ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

ohe.fit(
    X_train_s[nominal_cols]
)

ohe_train = pd.DataFrame(
    ohe.transform(
        X_train_s[nominal_cols]
    ),
    columns=ohe.get_feature_names_out(
        nominal_cols
    ),
    index=X_train_s.index
)

ohe_test = pd.DataFrame(
    ohe.transform(
        X_test_s[nominal_cols]
    ),
    columns=ohe.get_feature_names_out(
        nominal_cols
    ),
    index=X_test_s.index
)

X_train_s = pd.concat(
    [
        X_train_s[
            numeric_cols + binary_cols
        ],
        ohe_train
    ],
    axis=1
)

X_test_s = pd.concat(
    [
        X_test_s[
            numeric_cols + binary_cols
        ],
        ohe_test
    ],
    axis=1
)

print(
    "Training feature shape after encoding:",
    X_train_s.shape
)

print(
    "Testing feature shape after encoding:",
    X_test_s.shape
)

Training feature shape after encoding: (4088, 17)
Testing feature shape after encoding: (1022, 17)


## 5.8 Handling Missing Numerical Values

The Stroke Prediction Dataset contains missing numerical values, particularly in the BMI variable.

Mean imputation is applied to the numerical variables only.

The imputer is fitted on the training data and then applied to the test data.

In [18]:
imputer_stroke = SimpleImputer(
    strategy="mean"
)

X_train_s[numeric_cols] = (
    imputer_stroke.fit_transform(
        X_train_s[numeric_cols]
    )
)

X_test_s[numeric_cols] = (
    imputer_stroke.transform(
        X_test_s[numeric_cols]
    )
)

print(
    "Missing values in training data:",
    X_train_s.isnull().sum().sum()
)

print(
    "Missing values in testing data:",
    X_test_s.isnull().sum().sum()
)

Missing values in training data: 0
Missing values in testing data: 0


## 5.9 Feature Scaling

All predictor variables are standardised before being supplied to the SVM and MLP models.

The scaler is fitted only on the training data.

In [19]:
scaler_stroke = StandardScaler()

X_train_s_scaled = (
    scaler_stroke.fit_transform(
        X_train_s
    )
)

X_test_s_scaled = (
    scaler_stroke.transform(
        X_test_s
    )
)

print(
    "Scaled training shape:",
    X_train_s_scaled.shape
)

print(
    "Scaled testing shape:",
    X_test_s_scaled.shape
)

Scaled training shape: (4088, 17)
Scaled testing shape: (1022, 17)


## 5.10 Support Vector Machine (SVM) – Stroke

An RBF-kernel SVM is trained to predict stroke occurrence.

The Stroke dataset is highly imbalanced, so `class_weight="balanced"` is used to give greater importance to the minority stroke class during model training.

This is important because correctly identifying stroke cases is more informative than relying on accuracy alone.

In [20]:
svm_stroke = SVC(
    kernel="rbf",
    probability=True,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

svm_stroke.fit(
    X_train_s_scaled,
    y_train_s
)

pred_svm_s = svm_stroke.predict(
    X_test_s_scaled
)

proba_svm_s = svm_stroke.predict_proba(
    X_test_s_scaled
)

evaluate_model(
    "SVM on Stroke",
    y_test_s,
    pred_svm_s,
    proba_svm_s
)


--- SVM on Stroke ---
Accuracy : 0.772
Precision: 0.9327
Recall   : 0.772
F1 Score : 0.8345
ROC-AUC  : 0.7956
Confusion Matrix:
 [[759 213]
 [ 20  30]]

Full classification report:
               precision    recall  f1-score   support

           0       0.97      0.78      0.87       972
           1       0.12      0.60      0.20        50

    accuracy                           0.77      1022
   macro avg       0.55      0.69      0.54      1022
weighted avg       0.93      0.77      0.83      1022



## 5.11 Multi-Layer Perceptron (MLP) – Stroke

The same MLP architecture used for the BEED dataset is applied to the Stroke Prediction Dataset.

Using the same general architecture allows the performance of the neural network to be compared across the two healthcare datasets.

In [21]:
mlp_stroke = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    early_stopping=True,
    random_state=RANDOM_STATE
)

mlp_stroke.fit(
    X_train_s_scaled,
    y_train_s
)

pred_mlp_s = mlp_stroke.predict(
    X_test_s_scaled
)

proba_mlp_s = mlp_stroke.predict_proba(
    X_test_s_scaled
)

evaluate_model(
    "MLP (Neural Network) on Stroke",
    y_test_s,
    pred_mlp_s,
    proba_mlp_s
)


--- MLP (Neural Network) on Stroke ---
Accuracy : 0.9511
Precision: 0.9045
Recall   : 0.9511
F1 Score : 0.9272
ROC-AUC  : 0.5416
Confusion Matrix:
 [[972   0]
 [ 50   0]]

Full classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97       972
           1       0.00      0.00      0.00        50

    accuracy                           0.95      1022
   macro avg       0.48      0.50      0.49      1022
weighted avg       0.90      0.95      0.93      1022



# 6. Model Performance Comparison

The four trained models are compared using accuracy, precision, recall, F1-score and ROC-AUC.

The comparison is used to determine:

1. Which model performs best on the BEED dataset.
2. Which model performs best on the Stroke Prediction Dataset.
3. Whether the same algorithm performs consistently across different healthcare data types.
4. Whether the EEG and clinical datasets provide complementary insights.

In [22]:
results = []

models = [
    (
        "SVM",
        "BEED",
        y_test_b,
        pred_svm_b,
        proba_svm_b
    ),
    (
        "MLP",
        "BEED",
        y_test_b,
        pred_mlp_b,
        proba_mlp_b
    ),
    (
        "SVM",
        "Stroke",
        y_test_s,
        pred_svm_s,
        proba_svm_s
    ),
    (
        "MLP",
        "Stroke",
        y_test_s,
        pred_mlp_s,
        proba_mlp_s
    )
]

for (
    model_name,
    dataset_name,
    y_true,
    y_pred,
    y_proba
) in models:
    
    if y_proba.shape[1] == 2:
        auc = roc_auc_score(
            y_true,
            y_proba[:, 1]
        )
    else:
        auc = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr"
        )
    
    results.append({
        "Dataset": dataset_name,
        "Model": model_name,
        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),
        "Precision": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "ROC-AUC": auc
    })

results_df = pd.DataFrame(results)

results_df

,Dataset,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BEED,SVM,0.744375,0.773054,0.744375,0.731021,0.927222
1,BEED,MLP,0.941875,0.942755,0.941875,0.942208,0.993928
2,Stroke,SVM,0.772016,0.932698,0.772016,0.834538,0.795576
3,Stroke,MLP,0.951076,0.904546,0.951076,0.927228,0.541584


In [23]:
results_df.round(4)

,Dataset,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,BEED,SVM,0.7444,0.7731,0.7444,0.7310,0.9272
1,BEED,MLP,0.9419,0.9428,0.9419,0.9422,0.9939
2,Stroke,SVM,0.7720,0.9327,0.7720,0.8345,0.7956
3,Stroke,MLP,0.9511,0.9045,0.9511,0.9272,0.5416


## 6.1 Confusion Matrices

Confusion matrices are examined to understand the types of classification errors produced by each model.

This is particularly relevant for healthcare applications because false negatives can represent patients whose condition is not detected.

In [24]:
print("SVM - BEED")
print(
    confusion_matrix(
        y_test_b,
        pred_svm_b
    )
)

print("\nMLP - BEED")
print(
    confusion_matrix(
        y_test_b,
        pred_mlp_b
    )
)

print("\nSVM - Stroke")
print(
    confusion_matrix(
        y_test_s,
        pred_svm_s
    )
)

print("\nMLP - Stroke")
print(
    confusion_matrix(
        y_test_s,
        pred_mlp_s
    )
)

SVM - BEED
[[393   5   0   2]
 [  0 284  77  39]
 [  0  13 373  14]
 [  0  50 209 141]]

MLP - BEED
[[398   0   0   2]
 [  0 381   6  13]
 [  0   1 367  32]
 [  0   5  34 361]]

SVM - Stroke
[[759 213]
 [ 20  30]]

MLP - Stroke
[[972   0]
 [ 50   0]]


## 6.2 Detailed Classification Reports

Detailed classification reports are examined to identify class-specific precision, recall and F1-score.

This is particularly important for the Stroke dataset because the stroke class is substantially smaller than the non-stroke class.

In [25]:
print("=" * 70)
print("SVM - BEED")
print("=" * 70)

print(
    classification_report(
        y_test_b,
        pred_svm_b,
        zero_division=0
    )
)


print("=" * 70)
print("MLP - BEED")
print("=" * 70)

print(
    classification_report(
        y_test_b,
        pred_mlp_b,
        zero_division=0
    )
)


print("=" * 70)
print("SVM - Stroke")
print("=" * 70)

print(
    classification_report(
        y_test_s,
        pred_svm_s,
        zero_division=0
    )
)


print("=" * 70)
print("MLP - Stroke")
print("=" * 70)

print(
    classification_report(
        y_test_s,
        pred_mlp_s,
        zero_division=0
    )
)

SVM - BEED
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       400
           1       0.81      0.71      0.76       400
           2       0.57      0.93      0.70       400
           3       0.72      0.35      0.47       400

    accuracy                           0.74      1600
   macro avg       0.77      0.74      0.73      1600
weighted avg       0.77      0.74      0.73      1600

MLP - BEED
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       400
           1       0.98      0.95      0.97       400
           2       0.90      0.92      0.91       400
           3       0.88      0.90      0.89       400

    accuracy                           0.94      1600
   macro avg       0.94      0.94      0.94      1600
weighted avg       0.94      0.94      0.94      1600

SVM - Stroke
              precision    recall  f1-score   support

           0       0.97      0.78      0

# 7. Results and Discussion

## 7.1 Insights from the BEED Dataset

The BEED dataset provides EEG-based information for identifying epilepsy-related classes.

The model results will be examined to determine which algorithm provides the strongest classification performance and whether the EEG features contain sufficient information to distinguish between the target classes.

**Results to insert after model execution:**

- Best-performing model:
- Accuracy:
- Precision:
- Recall:
- F1-score:
- ROC-AUC:

---

## 7.2 Insights from the Stroke Dataset

The Stroke Prediction Dataset provides demographic and clinical risk factors associated with stroke.

The results will be interpreted with particular attention to recall and F1-score because the dataset contains substantially fewer stroke cases than non-stroke cases.

**Results to insert after model execution:**

- Best-performing model:
- Accuracy:
- Precision:
- Recall:
- F1-score:
- ROC-AUC:

---

## 7.3 Comparison of SVM and MLP

The SVM and MLP models will be compared across both datasets.

The comparison will consider whether one algorithm consistently performs better or whether model effectiveness depends on the characteristics of the healthcare dataset.

---

## 7.4 Comparison of the Two Data Sources

The BEED and Stroke datasets provide complementary forms of healthcare information.

BEED uses EEG-derived features associated with epilepsy, whereas the Stroke dataset uses demographic and clinical risk factors.

Therefore, the datasets represent different approaches to healthcare prediction:

- biomedical signal-based classification
- structured clinical risk prediction

---

## 7.5 Effectiveness of the Evaluation Metrics

Accuracy alone is insufficient for the Stroke dataset because of class imbalance.

Precision, recall and F1-score provide additional information about classification performance.

Recall is particularly important in healthcare because false-negative predictions may result in affected patients being incorrectly classified as not having the condition.

ROC-AUC provides an additional measure of the models' ability to discriminate between classes.

---

## 7.6 Overall Conclusion

The final conclusion will compare the performance of SVM and MLP across both datasets and discuss the implications of the findings for healthcare data science and clinical decision-support applications.